# [EURON 11기 Research] 11주차 복습과제 — MAE (정답 · 해설)
**Masked Autoencoders Are Scalable Vision Learners** (He et al., CVPR 2022)

이번 복습과제는 세 파트로 구성되어 있습니다.

| 파트 | 내용 | 비고 |
|---|---|---|
| Part 1 | 개념 확인 (단답·서술) | 마크다운 셀에 답 작성 |
| Part 2 | MAE 핵심 로직 직접 구현 (patchify · random masking · masked MSE) | `____` 빈칸 채우기, CPU로 실행 가능 |
| Part 3 | 사전학습된 MAE로 재구성 실험 (Hugging Face `ViTMAEForPreTraining`) | 4주차 ViT 실습(교재 10장)과 같은 FashionMNIST 파이프라인 사용 |
| (선택) Part 4 | 토의 질문 | 자유 서술 |

**제출 방법**
- 이 노트북의 **사본**을 만들어 빈칸을 채우고 모든 셀을 실행한 뒤 `.ipynb`로 제출해 주세요.
- 파일명 & Pull Request 제목: `Week11복습과제_김이화` / 라벨: `복습과제` / branch: `Week_11`
- 코드, 실행 결과, 주석이 모두 포함되도록 해 주세요.

> 런타임: Part 2는 CPU로 충분합니다. Part 3는 GPU(런타임 → 런타임 유형 변경 → T4)를 권장하지만 CPU로도 실행됩니다.

---
## Part 1. 개념 확인 — 예시 답안

**Q1.** (i) Architecture: CNN은 regular grid에서 동작해 mask token·positional embedding을 넣기 어려웠으나 ViT로 해소. (ii) Information density: 언어는 의미 밀도가 높지만 이미지는 공간적 중복이 커서 가려진 patch를 이웃으로 쉽게 복원 가능. (iii) Decoder 역할: 이미지 decoder는 저수준 pixel을 복원하므로 decoder 설계가 표현의 semantic level을 좌우. → (ii)는 **매우 높은 비율(75%)의 random masking**으로 이어짐: 중복을 줄여 외삽만으로는 풀 수 없는 task를 만듦.

**Q2.** ① 정확도: 사전학습 때만 보던 mask token이 실제 이미지엔 없어 생기는 train–deploy gap 제거 → linear probing 59.6 → 73.5. ② 효율: encoder가 25% patch만 처리 → 학습 FLOPs 3.3× 감소, wall-clock 2.8×(ViT-L)~4.1× 가속, 메모리 절감.

**Q3.** visible patch는 입력으로 이미 주어져 복원이 trivial하므로, 학습 신호를 "보이지 않는 내용 추론"에 집중시키기 위함(BERT와 동일 원리). 논문 각주: 결과 기반 선택이며, 모든 pixel에 loss를 주면 정확도가 약간(~0.5%) 하락.

**Q4.** 언어는 정보 밀도가 높아 15%만 가려도 어려운 task지만, 이미지는 중복이 커서 적게 가리면 주변 patch로 쉽게 메워짐 → high-level 이해 없이 풀리는 task가 됨. 75%처럼 높게 가려야 의미 있는 self-supervisory task가 됨.

**Q5.** linear separability만으로 표현 품질을 판단하면 안 된다. MAE 표현은 선형 분리성은 낮지만 non-linear head를 조금만 학습해도 강한 feature이며, linear probing은 transfer 성능과도 상관이 약하다.

---
## Part 2. MAE 핵심 로직 직접 구현

공식 코드(`facebookresearch/mae`의 `models_mae.py`)를 바탕으로, MAE에서 가장 중요한 세 함수를 직접 구현합니다.

1. `patchify` : 이미지 `(N, 3, H, W)` → patch 시퀀스 `(N, L, p*p*3)`
2. `random_masking` : noise를 argsort해서 patch를 섞고 앞의 `(1 - mask_ratio)`만 남기기
3. `masked_mse_loss` : **masked patch에서만** MSE 평균

`____` 부분을 채운 뒤 아래 확인용 셀이 모두 통과하는지 확인하세요.

In [ ]:
import torch
torch.manual_seed(0)

N, C, H, W, p = 2, 3, 224, 224, 16
imgs = torch.randn(N, C, H, W)
print('patch 개수 L =', (H // p) * (W // p))

### 2-1. patchify
`(N, 3, H, W)` → `(N, h, p, w, p, 3)` 순서로 재배열한 뒤 `(N, h*w, p*p*3)`로 펴 줍니다.

In [ ]:
def patchify(imgs, p=16):
    N, C, H, W = imgs.shape
    h, w = H // p, W // p
    x = imgs.reshape(N, C, h, p, w, p)
    x = torch.einsum('nchpwq->nhwpqc', x)      # 채널을 맨 뒤로
    x = x.reshape(N, h * w, p * p * C)          # (N, L, p*p*3)
    return x

patches = patchify(imgs, p)
print(patches.shape)   # 기대값: torch.Size([2, 196, 768])
assert patches.shape == (2, 196, 768)
# 첫 번째 patch는 이미지 왼쪽 위 16x16 영역과 같아야 합니다
assert torch.allclose(patches[0, 0], imgs[0, :, :16, :16].permute(1, 2, 0).reshape(-1))
print('patchify 통과!')

### 2-2. random_masking
- `len_keep = int(L * (1 - mask_ratio))`
- patch마다 `noise ~ U(0, 1)` → `argsort`하면 sample별 random permutation(`ids_shuffle`)
- `ids_restore = argsort(ids_shuffle)` : 원래 순서로 되돌리는 index
- `mask` : **0 = keep, 1 = remove**, 원래 patch 순서 기준

In [ ]:
def random_masking(x, mask_ratio):
    N, L, D = x.shape
    len_keep = int(L * (1 - mask_ratio))

    noise = torch.rand(N, L, device=x.device)          # noise in [0, 1]
    ids_shuffle = torch.argsort(noise, dim=1)          # small is keep, large is remove
    ids_restore = torch.argsort(ids_shuffle, dim=1)

    ids_keep = ids_shuffle[:, :len_keep]
    x_masked = torch.gather(x, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, D))

    mask = torch.ones([N, L], device=x.device)
    mask[:, :len_keep] = 0
    mask = torch.gather(mask, dim=1, index=ids_restore)   # unshuffle
    return x_masked, mask, ids_restore

x_masked, mask, ids_restore = random_masking(patches, mask_ratio=0.75)
print('visible patch:', x_masked.shape)            # 기대값: torch.Size([2, 49, 768])
print('masked 개수 :', mask.sum(dim=1))            # 기대값: tensor([147., 147.])
assert x_masked.shape == (2, 49, 768)
assert torch.all(mask.sum(dim=1) == 147)
# 남은 patch(x_masked)는 mask == 0 인 위치의 patch와 정확히 같아야 합니다
for n in range(N):
    kept = patches[n][mask[n] == 0]
    assert torch.allclose(torch.sort(kept.sum(-1)).values, torch.sort(x_masked[n].sum(-1)).values)
print('random_masking 통과!')

### 2-3. unshuffle 확인
decoder에서는 encoder 출력 뒤에 mask token을 붙이고 `ids_restore`로 원래 순서로 되돌립니다.
아래에서는 mask token 대신 **0 벡터**를 붙여, 되돌린 결과가 "visible 위치는 원래 patch, masked 위치는 0"이 되는지 확인합니다.

In [ ]:
N, L, D = patches.shape
mask_tokens = torch.zeros(N, L - x_masked.shape[1], D)
x_ = torch.cat([x_masked, mask_tokens], dim=1)
x_ = torch.gather(x_, dim=1, index=ids_restore.unsqueeze(-1).repeat(1, 1, D))

expected = patches * (1 - mask).unsqueeze(-1)
assert torch.allclose(x_, expected)
print('unshuffle 통과! visible patch가 원래 위치로 돌아왔습니다.')

### 2-4. masked MSE loss
1. patch별로 pixel 차이의 제곱 평균 → `(N, L)`
2. `mask`(1 = remove)를 곱해 **가려진 patch에서만** 평균

`norm_pix=True`이면 patch마다 평균·분산으로 정규화한 target을 사용합니다(논문 Table 1d의 *pixel (w/ norm)*).

In [ ]:
def masked_mse_loss(imgs, pred, mask, p=16, norm_pix=False):
    target = patchify(imgs, p)
    if norm_pix:
        mean = target.mean(dim=-1, keepdim=True)
        var = target.var(dim=-1, keepdim=True)
        target = (target - mean) / (var + 1.e-6) ** .5

    loss = (pred - target) ** 2
    loss = loss.mean(dim=-1)                 # [N, L], patch별 평균
    loss = (loss * mask).sum() / mask.sum()  # masked patch에서만 평균
    return loss

# 확인: visible patch만 틀리고 masked patch는 정답인 예측 -> loss는 0이어야 합니다
pred = patchify(imgs, p).clone()
pred[mask == 0] += 5.0
print('loss =', masked_mse_loss(imgs, pred, mask).item())   # 기대값: 0.0
assert masked_mse_loss(imgs, pred, mask).item() == 0.0
print('masked_mse_loss 통과!')

**Q6.** 위 확인 셀에서 visible patch를 크게 틀리게 예측했는데도 loss가 0이 된 이유를 쓰고, 이것이 Part 3의 재구성 이미지(`reconstruction`)에서 어떻게 드러날지 예상해 보세요.

> 답:

**Q6 (예시 답안).** mask가 0인(visible) 위치는 loss에 곱해져 0이 되므로, visible patch를 얼마나 틀리든 loss에 반영되지 않는다. 그래서 학습된 MAE는 visible 위치의 출력을 신경 쓰지 않고, `reconstruction`에서 visible 위치가 오히려 거칠게/블록처럼 보일 수 있다(논문 Fig. 2 각주).

---
## Part 3. 사전학습된 MAE로 재구성 실험

Hugging Face의 `ViTMAEForPreTraining`(`facebook/vit-mae-base`)을 불러와 FashionMNIST 이미지를 재구성해 봅니다.
데이터 전처리는 **4주차 ViT 실습(교재 10장 p.609~624)**과 같은 방식(1채널 → 3채널 복제, `AutoImageProcessor`의 mean/std로 정규화)을 사용합니다.

> 참고: 이 모델은 ImageNet으로 사전학습되었고 FashionMNIST는 본 적이 없습니다.

In [ ]:
!pip -q install transformers

import torch
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from transformers import AutoImageProcessor, ViTMAEForPreTraining

device = 'cuda' if torch.cuda.is_available() else 'cpu'
image_processor = AutoImageProcessor.from_pretrained('facebook/vit-mae-base')
model = ViTMAEForPreTraining.from_pretrained('facebook/vit-mae-base').to(device).eval()

print('mask_ratio   :', model.config.mask_ratio)
print('norm_pix_loss:', model.config.norm_pix_loss)
print('encoder      :', model.config.num_hidden_layers, 'blocks,', model.config.hidden_size, '-d')
print('decoder      :', model.config.decoder_num_hidden_layers, 'blocks,', model.config.decoder_hidden_size, '-d')

**Q7.** 위 출력에서 encoder와 decoder의 깊이·너비를 비교하고, 논문이 decoder를 이렇게 설계한 이유를 쓰세요.

> 답:

**Q7 (예시 답안).** encoder 12 blocks·768-d, decoder 8 blocks·512-d로 decoder가 더 얕고 좁다. decoder는 사전학습에서만 쓰이고 모든 token(196개)을 처리하므로, 가볍게 만들어 전체 연산을 줄이되(token당 연산 < encoder의 10%) linear probing을 위해 너무 얕지는 않게(8 blocks) 설계했다. (MAE-Base 기준; 논문 ablation은 ViT-L encoder 24 blocks·1024-d.)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize(size=(image_processor.size['height'], image_processor.size['width'])),
    transforms.Lambda(lambda x: torch.cat([x, x, x], 0)),        # 1채널 -> 3채널
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
])
test_dataset = datasets.FashionMNIST(root='./datasets', download=True, train=False, transform=transform)
classes = test_dataset.classes

pixel_values = torch.stack([test_dataset[i][0] for i in range(4)]).to(device)
print(pixel_values.shape)   # 기대값: torch.Size([4, 3, 224, 224])

In [ ]:
mean = torch.tensor(image_processor.image_mean).view(3, 1, 1)
std = torch.tensor(image_processor.image_std).view(3, 1, 1)

def to_img(t):
    # (3, H, W) 정규화 텐서 -> (H, W, 3) 이미지
    return (t.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0)

@torch.no_grad()
def reconstruct(pixel_values, mask_ratio=0.75, seed=0):
    model.config.mask_ratio = mask_ratio
    torch.manual_seed(seed)
    out = model(pixel_values)
    pred = model.unpatchify(out.logits)                        # (N, 3, H, W)
    p = model.config.patch_size
    mask = out.mask.unsqueeze(-1).repeat(1, 1, p * p * 3)      # (N, L, p*p*3)
    mask = model.unpatchify(mask)                              # 1 = remove, 0 = keep
    masked = pixel_values * (1 - mask)
    paste = pixel_values * (1 - mask) + pred * mask          # visible patch는 원본으로 덮어쓰기
    return out.loss.item(), masked, pred, paste

loss, masked, pred, paste = reconstruct(pixel_values, mask_ratio=0.75)
print(f'loss (mask 75%) = {loss:.4f}')

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for i in range(4):
    for j, (img, title) in enumerate([(pixel_values[i], 'original'), (masked[i], 'masked'),
                                      (pred[i], 'reconstruction'), (paste[i], 'recon + visible')]):
        axes[i, j].imshow(to_img(img)); axes[i, j].axis('off')
        if i == 0: axes[i, j].set_title(title)
plt.tight_layout(); plt.show()

### 3-2. masking ratio를 바꿔 보기
같은 이미지에 대해 masking ratio를 0.25 → 0.95로 바꿔 가며 loss와 재구성 결과를 비교합니다.

In [ ]:
ratios = [0.25, 0.5, 0.75, 0.9, 0.95]
losses = []
fig, axes = plt.subplots(2, len(ratios), figsize=(3 * len(ratios), 6))
for k, r in enumerate(ratios):
    loss, masked, pred, paste = reconstruct(pixel_values[:1], mask_ratio=r)
    losses.append(loss)
    axes[0, k].imshow(to_img(masked[0])); axes[0, k].set_title(f'mask {int(r*100)}%'); axes[0, k].axis('off')
    axes[1, k].imshow(to_img(paste[0])); axes[1, k].set_title(f'loss {loss:.3f}'); axes[1, k].axis('off')
plt.tight_layout(); plt.show()

plt.plot([int(r * 100) for r in ratios], losses, marker='o')
plt.xlabel('masking ratio (%)'); plt.ylabel('reconstruction loss (masked patch MSE)')
plt.title('Masking ratio vs. loss'); plt.grid(alpha=.3); plt.show()
model.config.mask_ratio = 0.75   # 원래 값으로 복구

**Q8.** masking ratio가 커질수록 loss와 재구성 결과는 어떻게 변했나요? 이 결과와 논문 Fig. 5(75%가 최적)를 연결해, "복원이 잘 되는 것"과 "좋은 표현을 배우는 것"이 같은지 다른지 의견을 쓰세요.

> 답:

**Q9.** `reconstruction` 열에서 visible patch 위치가 오히려 더 거칠게(블록처럼) 보였다면 그 이유는 무엇일까요? (Part 2의 Q6와 연결)

> 답:

**Q8 (예시 답안).** masking ratio가 커질수록 보이는 정보가 줄어 loss가 커지고 재구성이 흐려진다. 하지만 논문 Fig. 5에서는 25~50%보다 75%의 표현이 더 좋았다 → 복원 오차가 작다고 좋은 표현인 것은 아니며, 적당히 어려운 task가 의미 있는 표현 학습으로 이어진다. (95%처럼 너무 높으면 표현 품질도 다시 떨어짐.)

**Q9 (예시 답안).** loss를 masked patch에서만 계산하므로 visible 위치의 출력은 학습되지 않아 품질이 낮다. 그래서 `recon + visible`처럼 원본 visible patch로 덮어쓰면 자연스러워진다.

---
## (선택) Part 4. 토의 질문

1. 최적 masking ratio 75%는 데이터의 redundancy에 따라 달라질까요? 비디오·의료 영상·FashionMNIST처럼 배경이 단순한 이미지라면 어떨까요?
2. MAE의 "visible patch만 encoder에 넣기"는 ViT라서 가능한 설계입니다. CNN에 같은 아이디어를 적용하려면 무엇이 필요할까요?

> 답:

---
**참고 자료**
- He et al. (2022). *Masked Autoencoders Are Scalable Vision Learners*. CVPR. https://arxiv.org/abs/2111.06377
- 공식 코드: https://github.com/facebookresearch/mae (`models_mae.py`)
- Hugging Face ViTMAE 문서: https://huggingface.co/docs/transformers/model_doc/vit_mae
- 교재: 파이토치 트랜스포머를 활용한 자연어 처리와 컴퓨터비전 심층학습, 10장 ViT 실습 (p.609~624)